# 03 — Agentic AI Patterns Mapped to Customer Journey Stages

**Foundations | Mastering Agentic AI for Customer Journey Marketing**

Each of the six foundational agentic patterns maps naturally to a journey stage:

| Pattern | Journey Stage | Why It Fits |
|---------|--------------|-------------|
| Reflection | Awareness | Refine ad copy through self-critique |
| Tool Use | Consideration | Look up CRM/CDP data during nurture |
| Planning | Decision | Decompose deal progression into steps |
| Multi-Agent | Onboarding | Specialists collaborate on activation |
| Memory | Retention | Leverage interaction history for context |
| Guardrails | Advocacy | Validate review requests for brand safety |

In [ ]:
# File      : 03_agentic_patterns_by_stage.ipynb
# Stage     : Foundations
# Chapter   : 2
# Framework : All
# Author    : Pushparajan Ramar
# Repo      : https://github.com/Pushparajan/agenticai-marketing

import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
USE_MOCK = os.getenv("USE_MOCK_APIS", "true").lower() == "true"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY", "mock"), base_url=os.getenv("OPENAI_BASE_URL"))
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1")

def chat(messages: list[dict], **kw) -> str:
    if USE_MOCK:
        return f"[Mock response for: {messages[-1]['content'][:60]}...]"
    return client.chat.completions.create(model=MODEL, messages=messages, **kw).choices[0].message.content

print(f"Mock mode: {USE_MOCK}")

## Pattern 1: Reflection → Awareness Stage
Generate ad copy, self-critique, and refine — improving conversion before spend.

In [ ]:
def reflection_ad_copy(product: str, audience: str) -> dict:
    """Generate and refine ad copy through self-reflection."""
    draft = chat([{"role": "user", "content": f"Write a Google Ads headline + description for {product} targeting {audience}."}])
    critique = chat([{"role": "user", "content": f"Critique this ad copy for conversion: {draft}"}])
    refined = chat([{"role": "user", "content": f"Improve based on critique:\nDraft: {draft}\nCritique: {critique}"}])
    return {"draft": draft, "critique": critique, "refined": refined}

result = reflection_ad_copy("revenue forecasting SaaS", "B2B sales leaders")
for k, v in result.items():
    print(f"\n--- {k.upper()} ---\n{v}")

## Pattern 2: Tool Use → Consideration Stage
Look up CRM data to personalise nurture content during the consideration phase.

In [ ]:
CRM = {
    "alex@acme.com": {"name": "Alex Rivera", "company": "Acme Corp", "stage": "MQL", "score": 61, "interests": ["forecasting", "pipeline"]},
    "sam@bigco.com": {"name": "Sam Park", "company": "BigCo", "stage": "SQL", "score": 82, "interests": ["analytics", "reporting"]}
}

def crm_lookup(email: str) -> str:
    """Tool: look up contact in CRM."""
    return json.dumps(CRM.get(email, {"error": "not found"}))

def tool_use_nurture(email: str) -> str:
    """Use CRM data to personalise nurture content."""
    data = crm_lookup(email)
    prompt = f"Given this contact data, suggest a personalised nurture email topic:\n{data}"
    return chat([{"role": "user", "content": prompt}])

print(tool_use_nurture("alex@acme.com"))

## Pattern 3: Planning → Decision Stage
Decompose a complex deal progression into actionable steps.

In [ ]:
def plan_deal_progression(deal: dict) -> list[dict]:
    """Create a step-by-step deal progression plan."""
    if USE_MOCK:  # MOCK MODE
        return [
            {"step": 1, "action": "Send ROI calculator", "day": 0, "channel": "email"},
            {"step": 2, "action": "Address pricing objection", "day": 2, "channel": "call"},
            {"step": 3, "action": "Send competitor battlecard", "day": 4, "channel": "email"},
            {"step": 4, "action": "Executive sponsor intro", "day": 7, "channel": "meeting"},
            {"step": 5, "action": "Final proposal with urgency", "day": 10, "channel": "email"}
        ]
    raw = chat([{"role": "user", "content": f"Create a deal progression plan as JSON array for: {json.dumps(deal)}"}])
    return json.loads(raw)

plan = plan_deal_progression({"company": "Acme Corp", "value": 48000, "objections": ["pricing"], "competitor": "Clari"})
for s in plan:
    print(f"  Day {s['day']:2d} | [{s['channel']:7s}] {s['action']}")

## Pattern 4: Multi-Agent → Onboarding Stage
Multiple specialists collaborate on a customer activation plan.

In [ ]:
SPECIALISTS = {
    "product": "You are a product specialist. Identify the aha feature for this customer.",
    "success": "You are a CS manager. Design a 30-day activation timeline.",
    "training": "You are a training lead. Recommend learning resources."
}

def multi_agent_onboarding(customer: dict) -> dict:
    outputs = {}
    if USE_MOCK:  # MOCK MODE
        outputs["product"] = "Aha feature: Pipeline Forecasting Dashboard — 3x faster than spreadsheets."
        outputs["success"] = "Timeline: Day 1 data connect, Day 5 first report, Day 14 team rollout, Day 30 review."
        outputs["training"] = "Resources: Quick-start video (5min), forecasting webinar, API docs."
    else:
        context = json.dumps(customer)
        for role, prompt in SPECIALISTS.items():
            outputs[role] = chat([{"role": "system", "content": prompt}, {"role": "user", "content": context}])
            context += f"\n{role}: {outputs[role]}"
    return outputs

result = multi_agent_onboarding({"name": "Acme Corp", "plan": "Enterprise", "team_size": 12})
for role, output in result.items():
    print(f"\n{role.upper()}: {output}")

## Pattern 5: Memory → Retention Stage
Leverage full interaction history to detect issues and personalise retention outreach.

In [ ]:
class RetentionMemoryAgent:
    def __init__(self):
        self.history: list[dict] = [{"role": "system", "content": "You are a retention specialist. Use all past interaction context."}]

    def interact(self, event: str) -> str:
        self.history.append({"role": "user", "content": event})
        if USE_MOCK:  # MOCK MODE
            responses = [
                "Noted: customer onboarded successfully. Monitoring adoption.",
                "Usage declined 30%. Given their previous enthusiasm for forecasting, recommend a refresher session.",
                "NPS dropped to 6. Combined with usage decline, flagging as medium churn risk. Recommending exec outreach."
            ]
            resp = responses[min(len([h for h in self.history if h['role']=='user'])-1, 2)]
        else:
            resp = chat(self.history)
        self.history.append({"role": "assistant", "content": resp})
        return resp

agent = RetentionMemoryAgent()
events = ["Customer completed onboarding, adoption score 0.85", "Usage dropped to 0.42 over 30 days", "NPS survey: score 6"]
for e in events:
    print(f"\nEvent: {e}")
    print(f"Agent: {agent.interact(e)}")

## Pattern 6: Guardrails → Advocacy Stage
Validate review requests and referral invitations for brand safety and frequency limits.

In [ ]:
ADVOCACY_RULES = {
    "min_days_between_asks": 90,
    "banned_phrases": ["obligation", "must", "required to", "you owe"],
    "max_length": 300
}

def advocacy_guardrail(message: str, days_since_last_ask: int) -> dict:
    """Validate an advocacy request against brand and frequency rules."""
    issues = []
    if days_since_last_ask < ADVOCACY_RULES["min_days_between_asks"]:
        issues.append(f"Too soon — {days_since_last_ask}d since last ask (min {ADVOCACY_RULES['min_days_between_asks']}d)")
    for phrase in ADVOCACY_RULES["banned_phrases"]:
        if phrase.lower() in message.lower():
            issues.append(f"Banned phrase: '{phrase}'")
    if len(message) > ADVOCACY_RULES["max_length"]:
        issues.append(f"Too long ({len(message)} chars, max {ADVOCACY_RULES['max_length']})")
    return {"passed": len(issues) == 0, "issues": issues}

# Test: good request
good = advocacy_guardrail("We'd love your perspective on G2 — would you share a quick review?", 120)
print(f"Good request: passed={good['passed']}")

# Test: bad request
bad = advocacy_guardrail("You must leave a review. It's required to maintain your discount.", 30)
print(f"Bad request: passed={bad['passed']}, issues={bad['issues']}")

## Key Takeaways

| Pattern | Stage | Book Chapter |
|---------|-------|--------------|
| Reflection | Awareness | Ch 3 |
| Tool Use | Consideration | Ch 5 |
| Planning | Decision | Ch 7 |
| Multi-Agent | Onboarding | Ch 9 |
| Memory | Retention | Ch 10 |
| Guardrails | Advocacy | Ch 12 |

These patterns are the building blocks for every stage project.

**Next:** Stage 1 — Awareness with OpenAI Agents SDK →